# 04 · Final test on unseen turbulent wind fields
DDQN with the two-level adaptive rule (0.8/0.3 °/s) versus ROSCO at 11 and 15 m/s, TI = 10 %.

* Check that the test winds differ from the training and validation winds.
* Table 6 (`tab:final_test`).

Analysis window: 200–5200 s.

In [1]:
import numpy as np
import pandas as pd
from utils import load_fast_file, window, scaled_std, CHANNELS, T_MIN, T_MAX

# scenario -> (ROSCO file, DDQN file)
TEST = {
    "11 m/s TI=10% (test)": ("test_11ti10_rosco.out", "test_11ti10_ddqn_0.8-0.3dps.out"),
    "15 m/s TI=10% (test)": ("test_15ti10_rosco.out", "test_15ti10_ddqn_0.8-0.3dps.out"),
}
data = {name: (load_fast_file(ro)[0], load_fast_file(rl)[0]) for name, (ro, rl) in TEST.items()}

# Reference winds for the correlation check (ROSCO runs, same inflow as DDQN)
REFERENCE_WINDS = {
    "11 m/s training": "train_11ti10_rosco.out",
    "11 m/s validation": "val_11ti10_rosco.out",
    "15 m/s validation": "val_15ti10_rosco.out",
}

## Test winds

In [2]:
def correlation(a, b):
    t = np.arange(T_MIN, T_MAX, 0.025)
    return np.corrcoef(np.interp(t, a["Time"], a["Wind1VelX"]),
                       np.interp(t, b["Time"], b["Wind1VelX"]))[0, 1]

rows = []
for name, (rosco, ddqn) in data.items():
    u = window(rosco)["Wind1VelX"]
    rows.append({"Scenario": name, "Mean [m/s]": u.mean(), "TI [%]": 100 * u.std() / u.mean(),
                 "Same wind DDQN/ROSCO": np.allclose(window(rosco)["Wind1VelX"], window(ddqn)["Wind1VelX"])})
print(pd.DataFrame(rows).round(2).to_string(index=False))

print("\nCorrelation with training and validation winds")
refs = {k: load_fast_file(f)[0] for k, f in REFERENCE_WINDS.items()}
for name, (rosco, _) in data.items():
    for ref_name, ref in refs.items():
        print(f"  {name:22s} vs {ref_name:18s}: {correlation(rosco, ref):5.2f}")

            Scenario  Mean [m/s]  TI [%]  Same wind DDQN/ROSCO
11 m/s TI=10% (test)       11.03    9.66                  True
15 m/s TI=10% (test)       15.01    9.01                  True

Correlation with training and validation winds
  11 m/s TI=10% (test)   vs 11 m/s training   : -0.10
  11 m/s TI=10% (test)   vs 11 m/s validation : -0.15
  11 m/s TI=10% (test)   vs 15 m/s validation :  0.01
  15 m/s TI=10% (test)   vs 11 m/s training   :  0.11
  15 m/s TI=10% (test)   vs 11 m/s validation : -0.03
  15 m/s TI=10% (test)   vs 15 m/s validation :  0.05


## Results table

In [3]:
results = {}
for name, (rosco, ddqn) in data.items():
    ro, rl = window(rosco), window(ddqn)
    col = {}
    for ch in CHANNELS:
        col[f"{ch} mean DDQN"] = rl[ch].mean()
        col[f"{ch} mean ROSCO"] = ro[ch].mean()
        col[f"{ch} scaled STD DDQN"] = scaled_std(rl[ch], ro[ch])
        col[f"{ch} scaled STD ROSCO"] = scaled_std(ro[ch], ro[ch])
    col["Last time step DDQN [s]"] = ddqn["Time"].iloc[-1]
    results[name] = col
df_test = pd.DataFrame(results)
display(df_test.round(3))


,11 m/s TI=10% (test),15 m/s TI=10% (test)
GenPwr mean DDQN,3366.558,3373.916
GenPwr mean ROSCO,3350.707,3368.999
GenPwr scaled STD DDQN,0.064,0.058
GenPwr scaled STD ROSCO,0.092,0.126
TwrBsMyt mean DDQN,45534.362,30371.116
TwrBsMyt mean ROSCO,44943.933,30253.617
TwrBsMyt scaled STD DDQN,0.167,0.143
TwrBsMyt scaled STD ROSCO,0.115,0.125
RootMyb1 mean DDQN,6428.051,3917.740
RootMyb1 mean ROSCO,6349.796,3902.953
